# Full Fine-tuning of GPT-2 M on WebNLG (paper-exact baseline)

Baseline for the LoRA reproduction. **All 354.8M parameters of GPT-2 M are trained.**

- **Backbone:** GPT-2 Medium (all params trainable)
- **Task:** WebNLG **2017 Challenge** (release_v2.1)
- **Train:** **Seen categories only (10)** — true zero-shot setup matching the paper
- **Separator:** `<|SEP|>` added as a single special token
- **Metrics:** BLEU, NIST, METEOR, ROUGE-L, CIDEr (NLTK supplementary), and BLEU + Java METEOR + TER on Unseen / Seen / All (paper exact)

## Hyperparameters — exactly the paper's WebNLG FT recipe

```bash
python src/gpt2_ft.py \
    --train_batch_size 8 --grad_acc 1 --seq_len 512 \
    --lr 0.0002 --weight_decay 0.01 --adam_beta2 0.999 \
    --clip 0.0 --scheduler linear --warmup_step 500 \
    --max_epoch 5 --label_smooth 0.1 --random_seed 110
```

Loss is over the full sequence (no `-100` masking on prompt). Training in fp16. Inference: beam=10, length_penalty=0.8, no_repeat_ngram_size=4.

**Memory note:** batch=8 in fp16 on GPT-2 M needs ~14–16 GB GPU. If you OOM, set `CFG['grad_checkpointing'] = True` (costs ~25% extra time, halves activation memory).

Outputs go to `results/webnlg/full_ft_v2.1_paper/`.

## 1. Setup

In [1]:
# One-time installs (uncomment on first run)
# %pip install torch transformers accelerate sacrebleu nltk rouge-score pycocoevalcap tqdm
# import nltk; nltk.download('wordnet'); nltk.download('omw-1.4'); nltk.download('punkt'); nltk.download('punkt_tab')

In [2]:
import os, json, math, random, time
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import GPT2Tokenizer, GPT2LMHeadModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

from webnlg_loader import load_webnlg

SEED = 110   # matches paper's --random_seed 110 in src/gpt2_ft.py
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

c:\Users\weita\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [3]:
CFG = {
    'model_name': 'gpt2-medium',
    'data_dir': '../../data/webnlg/raw',
    'data_version': 'v2.1',          # paper's WebNLG 2017 challenge data
    'train_seen_only': True,         # paper's true zero-shot setup
    'sep_token': '<|SEP|>',          # added as a special token
    # --- Optimization (paper: src/gpt2_ft.py for WebNLG) ---
    'max_length': 512,               # --seq_len 512
    'batch_size': 8,                 # --train_batch_size 8 (was 4; user requested 8)
    'grad_accum': 1,                 # --grad_acc 1
    'epochs': 5,                     # --max_epoch 5
    'lr': 2e-4,                      # --lr 0.0002
    'weight_decay': 0.01,            # --weight_decay 0.01
    'adam_beta1': 0.9,
    'adam_beta2': 0.999,             # --adam_beta2 0.999
    'adam_eps': 1e-8,
    'warmup_steps': 500,             # --warmup_step 500
    'label_smoothing': 0.1,          # --label_smooth 0.1
    'grad_clip_norm': None,          # --clip 0.0
    'use_fp16': True,                # paper trains in fp16
    'mask_prompt': False,            # paper full-sequence loss
    'grad_checkpointing': False,     # set True if you OOM at batch=8
    # --- Inference ---
    'beam_size': 10,
    'length_penalty': 0.8,
    'no_repeat_ngram_size': 4,
    'max_new_tokens': 100,
    'out_dir': Path('../../results/webnlg/full_ft_v2.1_paper'),
    'save_full_model': False,
}
CFG['out_dir'].mkdir(parents=True, exist_ok=True)
print('Run dir:', CFG['out_dir'].resolve())

Run dir: C:\Users\weita\Desktop\deep_learning\final_proj\results\full_ft_v2.1_paper


## 2. Load WebNLG

In [4]:
train_raw, dev_raw, test_raw = load_webnlg(CFG['data_dir'], version=CFG['data_version'])
print(f"version={CFG['data_version']}  train={len(train_raw)}  dev={len(dev_raw)}  test={len(test_raw)}")
print('Sample:', train_raw[0])

version=v2.1  train=12876  dev=1619  test=1600
Sample: {'src': 'Aarhus_Airport : cityServed : "Aarhus, Denmark"', 'refs': ['The Aarhus is the airport of Aarhus, Denmark.', 'Aarhus Airport serves the city of Aarhus, Denmark.'], 'category': 'Airport'}


In [5]:
SEEN_CATS = {'Airport','Astronaut','Building','City','ComicsCharacter',
             'Food','Monument','SportsTeam','University','WrittenWork'}
if CFG['train_seen_only']:
    before = len(train_raw)
    train_raw = [r for r in train_raw if r.get('category', '') in SEEN_CATS]
    print(f'Filtered train: {before} -> {len(train_raw)} entries (Seen categories only)')

train_rows = [{'src': r['src'], 'tgt': t} for r in train_raw for t in r['refs']]
dev_rows   = dev_raw
test_rows  = test_raw
print(f'train pairs={len(train_rows)}  dev examples={len(dev_rows)}  test examples={len(test_rows)}')

Filtered train: 12876 -> 7791 entries (Seen categories only)
train pairs=20471  dev examples=1619  test examples=1600


## 3. Model (Full Fine-tuning — no LoRA)

Every parameter has `requires_grad=True`. AdamW will keep two momentum tensors per parameter, so memory ≈ 4× the model weights (params + grad + 2 optimizer states). Set `grad_checkpointing=True` in `CFG` if you OOM.

In [6]:
tokenizer = GPT2Tokenizer.from_pretrained(CFG['model_name'])
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(CFG['model_name'])
if CFG['grad_checkpointing']:
    model.gradient_checkpointing_enable()
    model.config.use_cache = False  # required when grad checkpointing is on

# Add `<|SEP|>` as a single special token (one BPE id) to mark the prompt/target boundary.
old_vocab = len(tokenizer)
n_added = tokenizer.add_special_tokens({'additional_special_tokens': [CFG['sep_token']]})
sep_id  = tokenizer.convert_tokens_to_ids(CFG['sep_token'])
model.resize_token_embeddings(len(tokenizer))
with torch.no_grad():
    e = model.get_input_embeddings().weight
    e[sep_id] = e[:old_vocab].mean(dim=0)
print(f'Added {n_added} special token; vocab {old_vocab} -> {len(tokenizer)}; SEP id = {sep_id}')

model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)  -- full FT')

c:\Users\weita\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Added 1 special token; vocab 50257 -> 50258; SEP id = 50257
Trainable: 354,824,192 / 354,824,192  (100.00%)  -- full FT


## 4. Tokenization & DataLoader

`<src> || <tgt><eos>`. Loss masking is controlled by `CFG['mask_prompt']`:
- `False` (default, paper recipe): label = full token sequence — model learns to predict the prompt as well as the target
- `True` (LoRA-notebook style): prompt tokens are replaced with `-100` so loss only counts the verbalization

In [7]:
SEP = CFG['sep_token']   # '<|SEP|>' — one special-token id

class WebNLGDataset(Dataset):
    def __init__(self, rows, tok, max_len, mask_prompt):
        self.rows, self.tok, self.max_len, self.mask_prompt = rows, tok, max_len, mask_prompt
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        prompt = r['src'] + SEP
        full   = prompt + r['tgt'] + self.tok.eos_token
        prompt_ids = self.tok(prompt, add_special_tokens=False)['input_ids']
        full_ids   = self.tok(full,   add_special_tokens=False, truncation=True, max_length=self.max_len)['input_ids']
        labels = list(full_ids)
        if self.mask_prompt:
            for j in range(min(len(prompt_ids), len(labels))):
                labels[j] = -100
        return {'input_ids': full_ids, 'labels': labels}

def collate(batch, pad_id):
    L = max(len(b['input_ids']) for b in batch)
    ids, attn, lbl = [], [], []
    for b in batch:
        pad = L - len(b['input_ids'])
        ids.append(b['input_ids'] + [pad_id] * pad)
        attn.append([1] * len(b['input_ids']) + [0] * pad)
        lbl.append(b['labels'] + [-100] * pad)   # padding always masked
    return {
        'input_ids':      torch.tensor(ids),
        'attention_mask': torch.tensor(attn),
        'labels':         torch.tensor(lbl),
    }

train_ds = WebNLGDataset(train_rows, tokenizer, CFG['max_length'], CFG['mask_prompt'])
train_loader = DataLoader(
    train_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=0,
    collate_fn=lambda b: collate(b, tokenizer.pad_token_id),
)
print(f'Train batches: {len(train_loader)}  (mask_prompt={CFG["mask_prompt"]}, sep={SEP!r})')

Train batches: 2559  (mask_prompt=False, sep='<|SEP|>')


## 5. Training (full fine-tuning)

In [8]:
optimizer = AdamW(
    model.parameters(),
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay'],
    betas=(CFG['adam_beta1'], CFG['adam_beta2']),
    eps=CFG['adam_eps'],
)
total_steps = max(1, (len(train_loader) // CFG['grad_accum']) * CFG['epochs'])
scheduler   = get_linear_schedule_with_warmup(optimizer, CFG['warmup_steps'], total_steps)
loss_fn     = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'], ignore_index=-100)

# Mixed precision (paper recipe)
use_amp = bool(CFG['use_fp16']) and DEVICE == 'cuda'
scaler  = torch.amp.GradScaler('cuda', enabled=use_amp)

log = {'loss': [], 'epoch_loss': []}
torch.cuda.reset_peak_memory_stats() if DEVICE == 'cuda' else None
t0 = time.time()
model.train()

for epoch in range(CFG['epochs']):
    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{CFG['epochs']}")
    optimizer.zero_grad()
    running = []
    for i, batch in enumerate(pbar):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
            shift_logits = out.logits[:, :-1, :].contiguous()
            shift_labels = batch['labels'][:, 1:].contiguous()
            loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        scaler.scale(loss / CFG['grad_accum']).backward()
        if (i + 1) % CFG['grad_accum'] == 0:
            if CFG['grad_clip_norm'] is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip_norm'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        running.append(loss.item())
        log['loss'].append(loss.item())
        pbar.set_postfix(loss=sum(running[-50:]) / min(50, len(running)))
    log['epoch_loss'].append(sum(running) / len(running))
    print(f"epoch {epoch+1} mean loss: {log['epoch_loss'][-1]:.4f}")

train_time = time.time() - t0
peak_mem = (torch.cuda.max_memory_allocated() / 1e9) if DEVICE == 'cuda' else 0.0
samples_per_sec = (len(train_ds) * CFG['epochs']) / train_time
print(f'Train time: {train_time:.1f}s  peak mem: {peak_mem:.2f} GB  throughput: {samples_per_sec:.2f} samples/s')

epoch 1/5:   0%|          | 0/2559 [00:00<?, ?it/s]c:\Users\weita\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\models\gpt2\modeling_gpt2.py:544: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
epoch 1/5: 100%|██████████| 2559/2559 [04:02<00:00, 10.57it/s, loss=1.82]


epoch 1 mean loss: 2.0538


epoch 2/5: 100%|██████████| 2559/2559 [03:58<00:00, 10.75it/s, loss=1.72]


epoch 2 mean loss: 1.7521


epoch 3/5: 100%|██████████| 2559/2559 [04:01<00:00, 10.61it/s, loss=1.67]


epoch 3 mean loss: 1.6877


epoch 4/5: 100%|██████████| 2559/2559 [03:59<00:00, 10.67it/s, loss=1.62]


epoch 4 mean loss: 1.6379


epoch 5/5: 100%|██████████| 2559/2559 [03:59<00:00, 10.67it/s, loss=1.59]

epoch 5 mean loss: 1.5975
Train time: 1201.0s  peak mem: 11.13 GB  throughput: 85.23 samples/s


In [9]:
with open(CFG['out_dir'] / 'train_log.json', 'w') as f:
    json.dump(log, f)
if CFG['save_full_model']:
    # Full state dict is ~1.4 GB for GPT-2 M. Off by default.
    torch.save(model.state_dict(), CFG['out_dir'] / 'gpt2m_fullft.pt')
    print('Saved full model ->', CFG['out_dir'] / 'gpt2m_fullft.pt')
else:
    print('Skipped full-model save (set CFG["save_full_model"]=True to keep it).')

Skipped full-model save (set CFG["save_full_model"]=True to keep it).


## 6. Generation on Test Set (beam search)

In [10]:
if CFG['grad_checkpointing']:
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
model.eval()

predictions, references, sources = [], [], []

# Reset peak-memory counter so inference_peak_gpu_gb measures *just* generation,
# not training. Then time the generation loop.
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
inf_t0 = time.time()

with torch.no_grad():
    for r in tqdm(test_rows, desc='generate'):
        prompt = r['src'] + SEP
        ids = tokenizer(prompt, return_tensors='pt').to(DEVICE)
        out = model.generate(
            **ids,
            max_new_tokens=CFG['max_new_tokens'],
            num_beams=CFG['beam_size'],
            do_sample=False,
            no_repeat_ngram_size=CFG['no_repeat_ngram_size'],
            length_penalty=CFG['length_penalty'],
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        gen = tokenizer.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        gen = gen.split('\n')[0].strip()
        predictions.append(gen)
        references.append(r['refs'])
        sources.append(r['src'])

inference_time     = time.time() - inf_t0
inference_peak_mem = (torch.cuda.max_memory_allocated() / 1e9) if DEVICE == 'cuda' else 0.0
inference_throughput = len(test_rows) / inference_time
print(f'Inference time: {inference_time:.1f}s  peak mem: {inference_peak_mem:.2f} GB  throughput: {inference_throughput:.2f} samples/s')

with open(CFG['out_dir'] / 'predictions.jsonl', 'w', encoding='utf-8') as f:
    for s, p, refs in zip(sources, predictions, references):
        f.write(json.dumps({'src': s, 'pred': p, 'refs': refs}) + '\n')
print('Wrote', len(predictions), 'predictions ->', CFG['out_dir'] / 'predictions.jsonl')

generate: 100%|██████████| 1600/1600 [11:01<00:00,  2.42it/s]

Inference time: 661.5s  peak mem: 5.78 GB  throughput: 2.42 samples/s
Wrote 1600 predictions -> ..\results\full_ft_v2.1_paper\predictions.jsonl


## 7. Evaluation: BLEU, NIST, METEOR, ROUGE-L, CIDEr, TER

TER (Translation Edit Rate) is the metric the LoRA paper reports for WebNLG (Table 14). Lower is better.

In [11]:
import sacrebleu
from nltk.translate.nist_score import corpus_nist
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from rouge_score import rouge_scorer

max_refs = max(len(r) for r in references)
refs_rect = [[r[i] if i < len(r) else r[0] for r in references] for i in range(max_refs)]
BLEU = float(sacrebleu.corpus_bleu(predictions, refs_rect).score)

# --- TER (sacrebleu corpus TER; lower is better; paper's WebNLG metric) ---
ter_obj = sacrebleu.corpus_ter([p if p.strip() else ' ' for p in predictions], refs_rect)
TER      = float(ter_obj.score)        # 0-100 (sacrebleu native)
TER_unit = TER / 100.0                  # 0-1   (paper's scale)

ref_tok = [[word_tokenize(t.lower()) for t in r] for r in references]
hyp_tok = [word_tokenize(p.lower()) if p else ['<empty>'] for p in predictions]
try:
    NIST = float(corpus_nist(ref_tok, hyp_tok, n=4))
except (ZeroDivisionError, ValueError):
    NIST = 0.0

m_scores = []
for refs, pred in zip(references, predictions):
    refs_w = [word_tokenize(t.lower()) for t in refs]
    pred_w = word_tokenize(pred.lower()) if pred else []
    if not pred_w:
        m_scores.append(0.0); continue
    m_scores.append(meteor_score(refs_w, pred_w))
METEOR = float(sum(m_scores) / max(1, len(m_scores)))

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rl = []
for refs, pred in zip(references, predictions):
    if not pred:
        rl.append(0.0); continue
    rl.append(max(scorer.score(r, pred)['rougeL'].fmeasure for r in refs))
ROUGE_L = float(sum(rl) / max(1, len(rl)))

print(f'BLEU    : {BLEU:.4f}')
print(f'NIST    : {NIST:.4f}')
print(f'METEOR  : {METEOR:.4f}')
print(f'ROUGE-L : {ROUGE_L:.4f}')
print(f'TER     : {TER:.4f}  ({TER_unit:.4f} on paper 0-1 scale, lower is better)')

BLEU    : 41.3041
NIST    : 7.5330
METEOR  : 0.6059
ROUGE-L : 0.5662
TER     : 60.1506  (0.6015 on paper 0-1 scale, lower is better)


In [12]:
from pycocoevalcap.cider.cider import Cider
gts = {i: list(refs)       for i, refs in enumerate(references)}
res = {i: [predictions[i]] for i in range(len(predictions))}
CIDEr, _ = Cider().compute_score(gts, res)
CIDEr = float(CIDEr)
print(f'CIDEr   : {CIDEr:.4f}')

CIDEr   : 2.1503


In [13]:
metrics = {
    'mode':     'full_ft',
    # Supplementary metrics on the All set (NLTK-based; use paper_metrics for paper-comparable numbers)
    'BLEU':     BLEU,
    'NIST':     NIST,
    'METEOR':   METEOR,        # NLTK + WordNet (inflated vs paper Java METEOR)
    'ROUGE-L':  ROUGE_L,
    'CIDEr':    CIDEr,
    'TER':      TER,           # 0-100 sacrebleu native
    'TER_unit': TER_unit,      # 0-1 paper scale (lower is better)
    'efficiency': {
        'trainable_params':                    trainable,
        'total_params':                        total,
        'trainable_pct':                       100 * trainable / total,
        # Training
        'train_time_sec':                      train_time,
        'train_peak_gpu_gb':                   peak_mem,
        'train_throughput_samples_per_sec':    samples_per_sec,
        # Inference
        'inference_time_sec':                  inference_time,
        'inference_peak_gpu_gb':               inference_peak_mem,
        'inference_throughput_samples_per_sec': inference_throughput,
    },
    'config': {k: (str(v) if isinstance(v, Path) else v) for k, v in CFG.items()},
}
with open(CFG['out_dir'] / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

{
  "mode": "full_ft",
  "BLEU": 41.304121152794664,
  "NIST": 7.533048931833894,
  "METEOR": 0.6059115428081022,
  "ROUGE-L": 0.5662219219399701,
  "CIDEr": 2.1502675416312185,
  "TER": 60.1505677176118,
  "TER_unit": 0.601505677176118,
  "efficiency": {
    "trainable_params": 354824192,
    "total_params": 354824192,
    "trainable_pct": 100.0,
    "train_time_sec": 1200.9589834213257,
    "train_peak_gpu_gb": 11.127168,
    "train_throughput_samples_per_sec": 85.22772335522085,
    "inference_time_sec": 661.5250718593597,
    "inference_peak_gpu_gb": 5.776388096,
    "inference_throughput_samples_per_sec": 2.418653605225956
  },
  "config": {
    "model_name": "gpt2-medium",
    "data_dir": "../data/raw",
    "data_version": "v2.1",
    "train_seen_only": true,
    "sep_token": "<|SEP|>",
    "max_length": 512,
    "batch_size": 8,
    "grad_accum": 1,
    "epochs": 5,
    "lr": 0.0002,
    "weight_decay": 0.01,
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_eps": 1e-08,

In [14]:
# --- Paper-style metrics: BLEU + Java METEOR + TER, split into Unseen / Seen / All ---
# Java METEOR (pycocoevalcap, official meteor-1.5.jar) is what the LoRA paper uses.
# Requires Java on PATH; if Java is missing run:  pip install install-jdk;
# python -c "import jdk; jdk.install('17')"  and re-import the notebook.
import os
JDK_DIR = os.path.expanduser('~/.jdks')
if os.path.isdir(JDK_DIR):
    candidates = sorted([d for d in os.listdir(JDK_DIR) if d.startswith('jdk-')])
    if candidates:
        java_home = os.path.join(JDK_DIR, candidates[-1])
        os.environ['JAVA_HOME'] = java_home
        os.environ['PATH'] = os.path.join(java_home, 'bin') + os.pathsep + os.environ.get('PATH', '')

from pycocoevalcap.meteor.meteor import Meteor

SEEN_CATS   = {'Airport','Astronaut','Building','City','ComicsCharacter',
               'Food','Monument','SportsTeam','University','WrittenWork'}
UNSEEN_CATS = {'Artist','Athlete','CelestialBody','MeanOfTransportation','Politician'}
src_to_cat  = {r['src']: r.get('category', '') for r in test_rows}

def paper_metrics(preds, refs):
    if not preds:
        return {'BLEU': float('nan'), 'METEOR': float('nan'), 'TER': float('nan'), 'n': 0}
    safe = [p if p.strip() else ' ' for p in preds]
    mx = max(len(r) for r in refs)
    rect = [[r[i] if i < len(r) else r[0] for r in refs] for i in range(mx)]
    bleu = float(sacrebleu.corpus_bleu(safe, rect).score)
    ter  = float(sacrebleu.corpus_ter(safe, rect).score) / 100.0
    gts = {i: list(refs[i])  for i in range(len(refs))}
    res = {i: [safe[i]]      for i in range(len(safe))}
    meteor, _ = Meteor().compute_score(gts, res)
    return {'BLEU': bleu, 'METEOR': float(meteor), 'TER': ter, 'n': len(preds)}

splits = {'Seen': ([], []), 'Unseen': ([], []), 'All': ([], [])}
for s, p, r in zip(sources, predictions, references):
    splits['All'][0].append(p); splits['All'][1].append(r)
    cat = src_to_cat.get(s, '')
    if cat in SEEN_CATS:
        splits['Seen'][0].append(p); splits['Seen'][1].append(r)
    elif cat in UNSEEN_CATS:
        splits['Unseen'][0].append(p); splits['Unseen'][1].append(r)

paper_split = {name: paper_metrics(p, r) for name, (p, r) in splits.items()}
for name, m in paper_split.items():
    print(f'{name:6s}  BLEU={m["BLEU"]:6.2f}  METEOR(Java)={m["METEOR"]:.4f}  TER={m["TER"]:.4f}  (n={m["n"]})')

existing = json.load(open(CFG['out_dir'] / 'metrics.json'))
existing['paper_metrics'] = {
    'note': 'BLEU + Java METEOR (pycocoevalcap, official meteor-1.5.jar) + TER, split by 2017 challenge categories',
    'seen_categories':   sorted(SEEN_CATS),
    'unseen_categories': sorted(UNSEEN_CATS),
    'splits': paper_split,
}
json.dump(existing, open(CFG['out_dir'] / 'metrics.json', 'w'), indent=2)
print('\nUpdated metrics.json with paper_metrics block.')

Seen    BLEU= 56.14  METEOR(Java)=0.4335  TER=0.4546  (n=967)
Unseen  BLEU= 14.04  METEOR(Java)=0.1728  TER=0.8498  (n=633)
All     BLEU= 41.30  METEOR(Java)=0.3307  TER=0.6015  (n=1600)

Updated metrics.json with paper_metrics block.


In [15]:
import winsound

for freq, dur in [(880, 150), (1108, 150), (1318, 300)]:
    winsound.Beep(freq, dur)
print('Done.')

Done.
